# Lab 07: Statistical Analysis of Evaluation Results

## Objectives
- Load and structure evaluation results
- Compute bootstrap confidence intervals
- Perform paired significance tests
- Calculate effect sizes (Cohen's d, Cliff's delta)
- Apply Holm-Bonferroni multiple comparison correction
- Create publication-quality visualizations
- Interpret statistical findings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print('Imports successful!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

## Step 1: Load Evaluation Data
Create realistic evaluation results for 5 models across 100 test cases.

In [ ]:
models = {'GPT-4o': {'mean': 0.82, 'std': 0.12}, 'Claude Sonnet': {'mean': 0.85, 'std': 0.10}, 'Claude Opus': {'mean': 0.87, 'std': 0.09}, 'Gemini Flash': {'mean': 0.78, 'std': 0.13}, 'Llama Scout': {'mean': 0.72, 'std': 0.15}}
n_test_cases = 100
data_list = []
for model_name, params in models.items():
    scores = np.random.normal(params['mean'], params['std'], n_test_cases)
    scores = np.clip(scores, 0, 1)
    for i, score in enumerate(scores):
        data_list.append({'model': model_name, 'test_case': i, 'score': score})
eval_df = pd.DataFrame(data_list)
print(f'Evaluation data shape: {eval_df.shape}')
print(f'Number of models: {eval_df["model"].nunique()}')
print(f'Summary: ', eval_df.groupby('model')['score'].mean().round(3).to_dict())

## Step 2: Bootstrap Confidence Intervals
Implement bootstrap CI with 10k resamples for robustness.

In [ ]:
def bootstrap_mean_ci(data: np.ndarray, n_bootstrap: int = 10000, ci: float = 0.95) -> Dict:
    bootstrap_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_bootstrap)]
    alpha = 1 - ci
    return {'mean': np.mean(data), 'std': np.std(data), 'ci_lower': np.percentile(bootstrap_means, alpha/2 * 100), 'ci_upper': np.percentile(bootstrap_means, (1 - alpha/2) * 100)}
bootstrap_results = {}
for model in eval_df['model'].unique():
    model_scores = eval_df[eval_df['model'] == model]['score'].values
    bootstrap_results[model] = bootstrap_mean_ci(model_scores)
ci_df = pd.DataFrame(bootstrap_results).T
print('Bootstrap Confidence Intervals (95%):')
print(ci_df.round(4))

## Step 3: Significance Testing
Paired permutation tests for all model pairs.

In [ ]:
def paired_permutation_test(g1: np.ndarray, g2: np.ndarray, n_perm: int = 10000) -> Tuple[float, float]:
    observed = np.mean(g1) - np.mean(g2)
    diffs = g1 - g2
    perm_diffs = [np.mean(diffs * np.random.choice([-1, 1], size=len(diffs))) for _ in range(n_perm)]
    return observed, np.mean(np.abs(perm_diffs) >= np.abs(observed))
from itertools import combinations
model_list = list(eval_df['model'].unique())
model_scores = {m: eval_df[eval_df['model'] == m]['score'].values for m in model_list}
pairwise_results = []
for m1, m2 in combinations(model_list, 2):
    diff, pval = paired_permutation_test(model_scores[m1], model_scores[m2])
    pairwise_results.append({'Model 1': m1, 'Model 2': m2, 'Mean Diff': diff, 'P-value': pval})
pairwise_df = pd.DataFrame(pairwise_results)
print('Paired Permutation Tests:')
print(pairwise_df.round(4))

## Step 4: Effect Sizes
Cohen's d and Cliff's delta for all pairs.

In [ ]:
def cohens_d(g1: np.ndarray, g2: np.ndarray) -> float:
    n1, n2 = len(g1), len(g2)
    pooled_std = np.sqrt(((n1-1)*np.var(g1,ddof=1) + (n2-1)*np.var(g2,ddof=1)) / (n1+n2-2))
    return (np.mean(g1) - np.mean(g2)) / pooled_std
def cliffs_delta(g1: np.ndarray, g2: np.ndarray) -> float:
    return np.mean(g1[:, None] > g2[None, :]) * 2 - 1
for i, row in pairwise_df.iterrows():
    g1, g2 = model_scores[row['Model 1']], model_scores[row['Model 2']]
    pairwise_df.loc[i, 'Cohens_d'] = cohens_d(g1, g2)
    pairwise_df.loc[i, 'Cliff_delta'] = cliffs_delta(g1, g2)
print('Effect Sizes:')
print(pairwise_df[['Model 1', 'Model 2', 'Cohens_d', 'Cliff_delta']].round(4))

## Step 5: Multiple Comparison Correction
Holm-Bonferroni correction for p-values.

In [ ]:
def holm_bonferroni(pvals: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    sorted_idx = np.argsort(pvals)
    m = len(pvals)
    corrected = np.ones(m)
    for i, idx in enumerate(sorted_idx):
        corrected[idx] = min(pvals[idx] * (m - i), 1.0)
    return corrected
pvals = pairwise_df['P-value'].values
corrected_p = holm_bonferroni(pvals)
pairwise_df['Corrected P'] = corrected_p
print('Multiple Comparison Correction:')
print(pairwise_df[['Model 1', 'Model 2', 'P-value', 'Corrected P']].round(6))

## Step 6: Visualization
Publication-quality figures with CI, significance, effect sizes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
models_sorted = sorted(bootstrap_results.items(), key=lambda x: x[1]['mean'], reverse=True)
y_pos = np.arange(len(models_sorted))
for i, (model, result) in enumerate(models_sorted):
    ax.errorbar(result['mean'], i, xerr=[[result['mean']-result['ci_lower']], [result['ci_upper']-result['mean']]], fmt='o', capsize=5)
ax.set_yticks(y_pos)
ax.set_yticklabels([m[0] for m in models_sorted])
ax.set_xlabel('Mean Score')
ax.set_title('A) Model Scores with 95% CI')
ax.grid(True, alpha=0.3)
ax = axes[0, 1]
ax.hist(pairwise_df['P-value'], bins=15, color='coral', edgecolor='black')
ax.axvline(0.05, color='red', linestyle='--', label='alpha=0.05')
ax.set_xlabel('P-value')
ax.set_title('B) P-value Distribution')
ax.legend()
ax = axes[1, 0]
ax.barh(range(len(pairwise_df)), np.abs(pairwise_df['Cohels_d']), color='green', alpha=0.7)
ax.set_xlabel('Cohen\'s d (absolute)')
ax.set_title('C) Effect Sizes')
ax = axes[1, 1]
uncorrected = pairwise_df['P-value'] < 0.05
corrected = pairwise_df['Corrected P'] < 0.05
x = np.arange(len(pairwise_df))
ax.bar(x - 0.2, uncorrected.astype(int), 0.4, label='Uncorrected')
ax.bar(x + 0.2, corrected.astype(int), 0.4, label='Bonferroni')
ax.set_ylabel('Significant')
ax.set_title('D) Multiple Comparison Correction')
ax.legend()
plt.tight_layout()
plt.savefig('statistical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Interpretation Guide
- Bootstrap CIs: Non-parametric confidence intervals (no distributional assumptions)
- P-values: Permutation-based, more robust than parametric tests
- Cohen's d: Small (0.2), Medium (0.5), Large (0.8)
- Holm-Bonferroni: Controls family-wise error rate across multiple tests